<a href="https://colab.research.google.com/github/quangminhho004-blip/UWB_RADAR/blob/main/notebooks/DATA_PREPARE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# DATA_PREPARE — pipeline dữ liệu của đồ án

Chạy **một lần**, khoảng 45 phút, **không cần GPU**.

```
Zenodo 5.7 GB
   -> giai nen 13 GB CSV  (external/mobivital/dataset/mobivital/tripod/, ban duy nhat)
   -> data/processed/by_user/*.npz     scripts/make_npz.py
   -> data/processed/windows/          scripts/make_windows.py
   -> ~2.7 GB len Google Drive
```

Sau notebook này, mọi thí nghiệm chỉ giải nén từ Drive (2 phút) rồi chạy.

Trong đây có chạy `prep_breath_final.py` của MobiVital để sinh `data_final/*.npy` —
không phải để dùng về sau, mà để **đối chiếu**: chứng minh `by_user/*.npz` chứa
đúng từng byte những gì code gốc đọc ra. Chi tiết ở mục 8.

| | | |
|---|---|---|
| repo | `quangminhho004-blip/UWB_RADAR` | code đồ án |
| upstream | `nesl/mobivital-public` | clone riêng, không có LICENSE |
| dataset | Zenodo `10.5281/zenodo.15022885` | `tripod.zip` 5.7 GB |


## 0. Môi trường

Cần ít nhất **25 GB trống**: zip 5.7 GB + CSV 13 GB + npz 3 GB.


In [ ]:
import os
import subprocess


def sh(command):
    """Chạy một lệnh shell, trả về stdout + stderr đã strip."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return (done.stdout + done.stderr).strip()


print(sh("df -h /content | tail -1"))


## 1. Google Drive

Nơi cất dữ liệu đã xử lý để mọi thí nghiệm sau dùng lại.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/mobivital"
os.makedirs(DRIVE, exist_ok=True)
print("cất kết quả vào", DRIVE)


## 2. Code

Clone repo đồ án và upstream MobiVital. `external/mobivital/` bị `.gitignore`
chặn nên phải clone riêng mỗi phiên.


In [ ]:
REPO = "/content/UWB_RADAR"

if os.path.exists(REPO + "/.git"):
    os.chdir(REPO)
    print(sh("git pull -q origin main && echo 'code đã mới nhất'"))
else:
    os.chdir("/content")
    sh("rm -rf " + REPO)
    print(sh("git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git " + REPO))
    print(sh("git clone -q https://github.com/nesl/mobivital-public.git " + REPO + "/external/mobivital"))
    print(sh("pip install -q einops"))

os.chdir(REPO)
print("đứng ở        :", os.getcwd())
print("commit của mình:", sh("git rev-parse --short HEAD"))
print("commit MobiVital:", sh("git -C external/mobivital rev-parse --short HEAD"))


## 3. Tải dataset từ Zenodo

`aria2c` chia 16 luồng. **Không dùng `wget`** — đo thật trên Colab:

| | tốc độ | 5.7 GB |
|---|---|---|
| `wget` | 0.7 MB/s | 2.3 giờ |
| `aria2c -x16` | 56 MB/s | 2 phút |

Zenodo bóp băng thông mỗi kết nối nên chia nhiều luồng ăn ngay.


In [ ]:
ZIP = "/content/tripod.zip"
ZIP_SIZE = 5700705593      # byte, từ Zenodo API

if os.path.exists(ZIP) and os.path.getsize(ZIP) == ZIP_SIZE:
    print("đã có, bỏ qua bước tải")
else:
    sh("apt-get install -qq -y aria2")
    out = sh("aria2c -x16 -s16 -k5M --summary-interval=0 --console-log-level=error "
             "-d /content -o tripod.zip "
             "https://zenodo.org/api/records/15022885/files/tripod.zip/content")
    for line in out.split("\n"):
        if "OK" in line or "ERR" in line:
            print(line)

size = os.path.getsize(ZIP)
print("%.2f GB   đúng kích thước Zenodo: %s" % (size / 1e9, size == ZIP_SIZE))


## 4. Giải nén — một bản CSV duy nhất

Giải nén thẳng vào thư mục MobiVital, đúng đường dẫn `prep_breath_final.py`
dòng 18 đòi. Không giữ bản thứ hai ở đâu.

```
external/mobivital/dataset/mobivital/tripod/   1874 CSV   <- ban duy nhat
data/                                          chi thu pipeline cua minh sinh ra
```

Zip đã có sẵn thư mục `tripod/` bên trong nên giải nén vào `.../mobivital/`,
không vào `.../mobivital/tripod/` — không thì lồng hai tầng.


In [ ]:
CSV_DIR = "external/mobivital/dataset/mobivital/tripod"

os.makedirs("external/mobivital/dataset/mobivital", exist_ok=True)
sh("unzip -q -o " + ZIP + " -d external/mobivital/dataset/mobivital/")

print("số file CSV:", sh("ls " + CSV_DIR + " | wc -l"))
print("dung lượng :", sh("du -sh " + CSV_DIR + " | cut -f1"))
print("file mẫu   :", sorted(os.listdir(CSV_DIR))[0])


## 5. Dọn chỗ cho pipeline gốc

`scripts/mobivital/setup_dataset.py` chỉ thêm hai thứ MobiVital không có:

1. thư mục lối tắt **vá 52 tên file lỗi thời** — bảng kết quả họ commit sẵn ra
   đời trước khi Zenodo đổi tên, 52 dòng ghi mốc tháng 10 còn bản hiện tại là
   tháng 12; `evaluate.py` dòng 28 mở file theo tên trong bảng, gặp là chết
2. `.git/info/exclude` — giấu `dataset/` và `data_final/` khỏi git của họ

Không đụng một dòng nào trong code MobiVital.


In [ ]:
print(sh("python scripts/mobivital/setup_dataset.py"))


## 6. → `data_final/*.npy` — chạy chính script của MobiVital

Lệnh đầu tiên trong README của họ, nguyên bản:

```
CSV trong dataset/mobivital/tripod/
├─ loc ABCDEFKL -> data_final/training_breath_tripod_data.npy   1289 buoi ghi
└─ loc GHIJ     -> data_final/testing_breath_tripod_data.npy     537 buoi ghi
```

`prep_breath_final.py` dòng 5 `import matplotlib` (không dùng tới) — Colab có sẵn.


In [ ]:
print(sh("cd external/mobivital && python dataset_preparation/prep_breath_final.py 2>&1 | tail -2"))
print(sh("ls -la external/mobivital/data_final/"))
print()
print("KIỂM TRA không sửa gì trong repo MobiVital:")
print(sh("git -C external/mobivital status --short") or "  git status trống")


## 7. → `by_user/*.npz` — pipeline của mình

`scripts/make_npz.py` đọc đúng bộ CSV đó, gom theo từng người. Lọc người bằng
tên file, đúng cách `prep_breath_final.py` dòng 27-31 làm.

```
tripod/*.csv  ->  data/processed/by_user/A.npz … L.npz
                  uwb   (so buoi ghi, 1500, 120) complex64
                  gt    (so buoi ghi, 1500)      float32, chuan hoa [-1, 1]
                  files (so buoi ghi,)           ten file CSV
```


In [ ]:
print(sh("python scripts/make_npz.py"))


## 8. Đối chiếu — `by_user` == `data_final`, từng byte

Cùng một bộ CSV, hai đường đọc khác nhau. `scripts/check_data.py` ghép cặp từng
buổi ghi bằng chữ ký md5 của `gt` (hai bên xếp thứ tự khác nhau) rồi so cả `gt`
lẫn `uwb`. Phải in ra:

```
ABCDEFKL  1289/1289 buoi ghi khop TUNG BYTE   = training_breath_tripod_data.npy
GHIJ       537/537  buoi ghi khop TUNG BYTE   = testing_breath_tripod_data.npy
```

So bằng byte chứ không bằng sai số: cùng file CSV, cùng công thức, cùng
`float32` thì phải giống tuyệt đối. Khớp thì từ đây mọi thí nghiệm chỉ đọc
`by_user/*.npz`, bỏ được CSV thô 13 GB. Không khớp thì script tự dừng.


In [ ]:
print(sh("python scripts/check_data.py"))


## 9. → `windows/` — cắt sẵn cửa sổ để train

```
200 mau vao -> 25 mau phai doan, truot 25
```

`scripts/make_windows.py` gọi `generate_dataset` của MobiVital, cắt hai bộ:

```
dev_cv/       cat RIENG tung nguoi A B C D E F K L   -> run_cv.py ghep fold tuy y
final_train/  cat GOP 8 nguoi, doc thang data_final  -> run_final_test.py
```

Cửa sổ **chỉ để train**. Lúc chấm điểm đọc buổi ghi thô từ `by_user/*.npz`, vì
việc chọn sóng ở bước cắt này có nhìn nhịp thở thật (`corr > 0.9`) — nhìn vào lúc
chấm là rò rỉ.


In [ ]:
print(sh("python scripts/make_windows.py"))
print()
print(sh("du -sh data/processed/windows/dev_cv data/processed/windows/final_train"))


## 10. Băm nội dung — `results/checksums.txt`

`scripts/checksums.py` băm **nội dung mảng** (không băm vỏ ZIP — ZIP nhúng thời
điểm ghi nên hai file nội dung y hệt vẫn khác md5). Ai chạy lại notebook này ở
máy khác thì đối chiếu với `results/checksums.txt` đã commit trong repo.


In [ ]:
print(sh("python scripts/checksums.py"))
print()
print(sh("git diff --stat results/checksums.txt") or "  khớp checksums.txt đã commit")


## 11. Cất lên Drive

| | dùng để | ~ |
|---|---|---|
| `windows.tar.gz` | train | 100 MB |
| `by_user.tar` | chấm điểm | 2.6 GB |

Không đưa lên: CSV thô 13 GB (tải lại Zenodo 2 phút), `data_final/*.npy` (sinh lại
bằng script MobiVital).


In [ ]:
sh("tar -czf " + DRIVE + "/windows.tar.gz -C data/processed windows")
sh("tar -cf  " + DRIVE + "/by_user.tar    -C data/processed by_user")

print(sh("ls -la " + DRIVE))
print()
print("Drive đang dùng:", sh("du -sh " + DRIVE + " | cut -f1"))


## Xong

Mọi notebook thí nghiệm sau bắt đầu bằng ô này (~2 phút):

```python
from google.colab import drive
drive.mount("/content/drive")

!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!git clone -q https://github.com/nesl/mobivital-public.git external/mobivital
!pip install -q einops

!mkdir -p data/processed
!tar -xzf /content/drive/MyDrive/mobivital/windows.tar.gz -C data/processed/
!tar -xf  /content/drive/MyDrive/mobivital/by_user.tar    -C data/processed/

!ln -s /content/drive/MyDrive/mobivital/runs runs
```

`runs/` trỏ thẳng vào Drive — Colab ngắt phiên thì checkpoint vẫn còn, phiên sau
`resume` chạy tiếp.

**TN0 không cần bước này** — nó tự dựng lại toàn bộ dữ liệu từ đầu để đứng độc lập.
